# 03 Policy Gradient Training — Connect 4

This notebook implements the **REINFORCE** (policy gradient) algorithm to improve
`dean_cnn` (M1), the strongest baseline model from the round robin (76% win rate).

**Algorithm outline (Steps 2–3 from the project spec):**
1. Pick M1 = `dean_cnn`. Clone it so the original is preserved as a frozen reference.
2. Maintain an opponent pool starting from all other baseline models.
3. Each outer iteration:
   - Randomly pick M2 from the opponent pool.
   - Play `GAMES_PER_ITER` games of M1 vs M2, randomly assigning who goes first.
   - Both M1 and M2 sample moves stochastically from their model output (per spec).
   - Collect M1's `(board, move)` trajectory; compute discounted returns.
   - Normalize returns across the batch for training stability.
   - Sample a **fixed batch** of 64 triplets and take one gradient step.
4. Every `SNAPSHOT_EVERY` iterations, add a frozen copy of the current M1 to the
   opponent pool so M1 learns to beat progressively stronger versions of itself.
5. Every `EVAL_EVERY` iterations, record M1's greedy win rate against fixed
   reference opponents to track improvement.

**Prerequisites:** Run `01_import_models.ipynb` and `02_round_robin.ipynb` first.

In [1]:
import os
import sys
import random
import warnings
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
warnings.filterwarnings("ignore")

# Project root is one level above the notebooks/ folder
ROOT = Path.cwd().resolve().parent
SRC  = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

import tensorflow as tf
tf.get_logger().setLevel("ERROR")

from connect4_model_hub import (
    MODELS_DIR,
    DATA_DIR,
    LoadedBot,
    load_standardized_bots,
    legal_moves,
    apply_move,
    check_winner,
    encode_board_for_model,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print("TensorFlow:", tf.__version__)
print("Project root:", ROOT)

TensorFlow: 2.21.0
Project root: C:\Users\conno\OneDrive - The University of Texas at Austin\Documents\claude-projects\Optimization 2\Project 3


## Configuration

All hyperparameters are collected here so they are easy to tune.

In [2]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
SEED           = 42    # global RNG seed for reproducibility

# Models to use
LOAD_MODELS    = ["dean_cnn", "archie_resnet", "connor_cnn", "zac_cnn_best"]
M1_NAME        = "dean_cnn"   # the model being trained

# Game collection
GAMES_PER_ITER = 20    # games to play per outer iteration
OPENING_MIN    = 2     # min random opening moves per game
OPENING_MAX    = 4     # max random opening moves per game
# No temperature — both M1 and M2 sample from raw softmax per spec

# Training
GAMMA          = 0.95  # discount factor for computing returns
BATCH_SIZE     = 64    # MUST stay constant — TF retraces the graph on shape changes!
LR             = 1e-5  # Adam learning rate (small — fine-tuning a pre-trained model)
NUM_ITERATIONS = 600   # total outer iterations (~12 000 games total)

# Opponent pool management
MAX_POOL_SIZE  = 15    # hard cap on total pool size (originals never removed)
SNAPSHOT_EVERY = 75    # add a frozen M1 copy to opponent pool every N iters

# Evaluation
EVAL_EVERY     = 25    # evaluate greedy win rate every N iters
EVAL_GAMES     = 20    # greedy games per eval opponent (half as first player, half second)

# Output paths
PG_MODEL_PATH  = MODELS_DIR / "pg_m1_trained.keras"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Seed everything
np.random.seed(SEED)
tf.random.set_seed(SEED)
rng = random.Random(SEED)

print("Config loaded.")
print(f"  Models in use      : {LOAD_MODELS}")
print(f"  Total games (approx): {NUM_ITERATIONS * GAMES_PER_ITER:,}")
print(f"  Snapshot schedule  : every {SNAPSHOT_EVERY} iters, pool cap {MAX_POOL_SIZE}")
print(f"  Model will be saved to: {PG_MODEL_PATH}")

Config loaded.
  Models in use      : ['dean_cnn', 'archie_resnet', 'connor_cnn', 'zac_cnn_best']
  Total games (approx): 12,000
  Snapshot schedule  : every 75 iters, pool cap 15
  Model will be saved to: C:\Users\conno\OneDrive - The University of Texas at Austin\Documents\claude-projects\Optimization 2\Project 3\models\pg_m1_trained.keras


## Load Models and Build Opponent Pool

In [3]:
# Fix manifest paths to point to this machine
_manifest_path = MODELS_DIR / "manifest.csv"
_manifest = pd.read_csv(_manifest_path)
_manifest["standardized_path"] = _manifest["name"].apply(
    lambda name: str(MODELS_DIR / f"{name}.keras")
)
_manifest.to_csv(_manifest_path, index=False)
print("Manifest paths updated.")

Manifest paths updated.


In [4]:
# Load only the 4 specified models
all_bots    = load_standardized_bots(MODELS_DIR)
all_bots    = [b for b in all_bots if b.name in LOAD_MODELS]
bot_by_name = {b.name: b for b in all_bots}

m1_ref_bot    = bot_by_name[M1_NAME]
M1_INPUT_SHAPE = m1_ref_bot.input_shape

print(f"Loaded {len(all_bots)} models:")
for b in all_bots:
    print(f"  {b.name}  {b.input_shape}")
print(f"\nM1 = {M1_NAME}  |  input shape: {M1_INPUT_SHAPE}")

Loaded 4 models:
  archie_resnet  (None, 6, 7, 2)
  connor_cnn  (None, 6, 7, 1)
  dean_cnn  (None, 6, 7, 2)
  zac_cnn_best  (None, 6, 7, 2)

M1 = dean_cnn  |  input shape: (None, 6, 7, 2)


In [5]:
# Clone M1's architecture AND weights into a fresh trainable model.
m1_model = tf.keras.models.clone_model(m1_ref_bot.model)
m1_model.set_weights(m1_ref_bot.model.get_weights())

# Compile the call method into a static TF graph so repeated single-board
# inference during game play doesn't re-enter Python each time.
m1_model.call = tf.function(m1_model.call, experimental_relax_shapes=True)

print("Trainable M1 clone created.")
m1_model.summary()

Trainable M1 clone created.


Model: "DeepResNet_Connect4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 6, 7, 2)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 6, 7, 128) │      2,432 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 6, 7, 128) │        512 │ conv2d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ elu (ELU)           │ (None, 6, 7, 128) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 6, 7, 128) │    147,584 │ elu[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 6, 7, 128) │        512 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ elu_1 (ELU)         │ (None, 6, 7, 128) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 6, 7, 128) │    147,584 │ elu_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 6, 7, 128) │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 6, 7, 128) │          0 │ elu[0][0],        │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ elu_2 (ELU)         │ (None, 6, 7, 128) │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 6, 7, 128) │    147,584 │ elu_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 6, 7, 128) │        512 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ elu_3 (ELU)         │ (None, 6, 7, 128) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 6, 7, 128) │    147,584 │ elu_3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 6, 7, 128) │        512 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 6, 7, 128) │          0 │ elu_2[0][0],      │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ elu_4 (ELU)         │ (None, 6, 7, 128) │          0 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 6, 7, 128) │    147,584 │ elu_4[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 6, 7, 128) │        512 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 7,524,487 (28.70 MB)

 Trainable params: 7,518,599 (28.68 MB)

 Non-trainable params: 5,888 (23.00 KB)

In [6]:
# Original opponents = the 3 non-M1 CNNs + the frozen original M1 itself
# The spec says "the original version of M1 can also be on the list of opponents"
# None of these are ever removed from the pool
ORIGINAL_OPPONENTS = [b for b in all_bots if b.name != M1_NAME] + [m1_ref_bot]
opponent_pool      = list(ORIGINAL_OPPONENTS)

# Evaluation uses all original opponents
EVAL_OPPONENTS = list(ORIGINAL_OPPONENTS)

print("Original opponents (never removed):")
for b in ORIGINAL_OPPONENTS:
    print(f"  {b.name}")
print(f"\nPool cap: {MAX_POOL_SIZE} total  ({MAX_POOL_SIZE - len(ORIGINAL_OPPONENTS)} snapshot slots)")

Original opponents (never removed):
  archie_resnet
  connor_cnn
  zac_cnn_best
  dean_cnn

Pool cap: 15 total  (11 snapshot slots)


In [7]:
def get_winning_moves(board: np.ndarray, player: int) -> list:
    """Return columns where `player` wins immediately on the next move."""
    return [
        col for col in legal_moves(board)
        if check_winner(apply_move(board, col, player)) == player
    ]


def get_non_loser_moves(board: np.ndarray, player: int) -> list:
    """
    Return legal moves that do NOT give the opponent an immediate winning response.
    If every legal move leads to an opponent win, fall back to all legal moves
    (forced loss — play anything).
    """
    opponent = -player
    safe = []
    for col in legal_moves(board):
        next_board = apply_move(board, col, player)
        if not get_winning_moves(next_board, opponent):
            safe.append(col)
    return safe if safe else legal_moves(board)


def _softmax_over_legal(probs: np.ndarray, legal: list) -> np.ndarray:
    """
    Clip, restrict to legal moves, and renormalize.
    Falls back to uniform if all legal probabilities are zero (e.g. degenerate
    model output after many gradient updates).
    """
    legal_probs = np.clip(probs[legal], 0, None)
    total = legal_probs.sum()
    if total < 1e-10:
        return np.ones(len(legal), dtype=np.float64) / len(legal)
    return legal_probs / total


def m2_select_move(bot: LoadedBot, board: np.ndarray, player: int,
                   rng: random.Random) -> int:
    """
    M2 move selection: sample stochastically from the model's output distribution
    over legal moves, as required by the project spec.
    """
    legal   = legal_moves(board)
    encoded = encode_board_for_model(board, player, bot.input_shape)
    raw     = bot.model(encoded, training=False)
    probs   = np.asarray(raw, dtype=np.float64).reshape(-1)
    return int(np.random.choice(legal, p=_softmax_over_legal(probs, legal)))


def m1_sample_move(model, input_shape: tuple, board: np.ndarray, player: int):
    """
    Sample a legal move for M1 from raw softmax output — identical to M2's
    sampling, as required by the spec.

    Returns:
        col     : int — the column chosen
        encoded : np.ndarray shape (6, 7, 2) — the board state BEFORE the move
    """
    legal   = legal_moves(board)
    encoded = encode_board_for_model(board, player, input_shape)
    raw     = model(encoded, training=False)
    probs   = np.asarray(raw, dtype=np.float64).reshape(-1)
    col     = int(np.random.choice(legal, p=_softmax_over_legal(probs, legal)))
    return col, encoded[0]

### Game simulation

In [8]:
def play_pg_game(
    m1_model,
    m1_input_shape: tuple,
    m2_bot: LoadedBot,
    rng: random.Random,
    opening_min: int = 2,
    opening_max: int = 4,
):
    """
    Play one game of M1 vs M2 and collect M1's move trajectory.

    Both M1 and M2 sample moves from raw softmax output over legal moves,
    as required by the project spec.  Opening moves are played uniformly
    at random (from both sides) to diversify starting positions; these are
    NOT recorded in the trajectory.

    Returns:
        trajectory : list of (encoded_board, col) tuples — one per M1 move.
                     encoded_board has shape (6, 7, 2), no batch dimension.
        outcome    : +1 M1 won  |  -1 M1 lost  |  0 draw
    """
    board          = np.zeros((6, 7), dtype=np.int8)
    m1_player      = rng.choice([1, -1])
    m2_player      = -m1_player
    current_player = 1
    move_count     = 0
    opening_moves  = rng.randint(opening_min, opening_max)
    trajectory     = []

    while True:
        legal = legal_moves(board)
        if not legal:
            return trajectory, 0   # board full → draw

        # ── Opening phase: random moves, no learning ──────────────────────────
        if move_count < opening_moves:
            col = rng.choice(legal)

        # ── M1's turn: sample from raw softmax and record (board, col) ────────
        elif current_player == m1_player:
            col, enc = m1_sample_move(m1_model, m1_input_shape, board, m1_player)
            trajectory.append((enc, col))

        # ── M2's turn: sample from raw softmax ────────────────────────────────
        else:
            col = m2_select_move(m2_bot, board, m2_player, rng)

        board       = apply_move(board, col, current_player)
        move_count += 1

        winner = check_winner(board)
        if winner is not None:
            outcome = 1 if winner == m1_player else -1
            return trajectory, outcome

        current_player *= -1

### Discounted returns and batch builder

In [9]:
def compute_discounted_returns(
    trajectory_length: int, outcome: int, gamma: float = 0.95
) -> np.ndarray:
    """
    Compute the discounted return G_t for each step in M1's trajectory.

    The only reward is at the terminal state (win +1, loss -1, draw 0).
    Returns are discounted backwards so that moves closer to the outcome
    receive a larger-magnitude gradient signal:

        G_t = gamma^(T - t) * outcome

    where T is the index of M1's last move.
    """
    if trajectory_length == 0:
        return np.array([], dtype=np.float32)
    T      = trajectory_length - 1
    powers = np.arange(T, -1, -1, dtype=np.float32)   # [T, T-1, ..., 0]
    return (gamma ** powers) * float(outcome)


def build_training_batch(
    episode_batch: list,
    batch_size: int,
    gamma: float,
):
    """
    Pool all (board, col, G_t) triplets from the episode batch, then
    sample EXACTLY `batch_size` of them with replacement.

    WHY fixed batch size?
    tf.function traces a new graph for each unique input shape.  Passing a
    variable number of rows each step causes repeated retracing and makes
    training drastically slower.  Sampling to a constant size prevents this.

    Returns None if no M1 moves were collected (e.g. game ended in opening).
    Returns (boards, cols, returns) arrays of shape (batch_size, ...).
    """
    boards_list, cols_list, returns_list = [], [], []

    for trajectory, outcome in episode_batch:
        if not trajectory:
            continue
        Gs = compute_discounted_returns(len(trajectory), outcome, gamma)
        for (enc, col), G in zip(trajectory, Gs):
            boards_list.append(enc)
            cols_list.append(col)
            returns_list.append(G)

    if not boards_list:
        return None

    boards  = np.array(boards_list,  dtype=np.float32)   # (N, 6, 7, 2)
    cols    = np.array(cols_list,    dtype=np.int32)      # (N,)
    returns = np.array(returns_list, dtype=np.float32)    # (N,)

    # Sample with replacement to enforce constant batch size
    idx = np.random.choice(len(boards), size=batch_size, replace=True)
    return boards[idx], cols[idx], returns[idx]

### Policy gradient training step

The **REINFORCE** update rule:

$$\mathcal{L} = -\mathbb{E}\left[ G_t \cdot \log \pi(a_t \mid s_t) \right]$$

- Positive $G_t$ (win): increase the probability of the move that led here.
- Negative $G_t$ (loss): decrease the probability of the move that led here.

The `@tf.function` decorator compiles the step into a static graph.  Because
`BATCH_SIZE` is constant, TF traces the graph exactly once.

In [10]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LR)

@tf.function
def pg_train_step(
    boards: tf.Tensor,   # (BATCH_SIZE, 6, 7, 2)  float32
    cols: tf.Tensor,     # (BATCH_SIZE,)           int32
    returns: tf.Tensor,  # (BATCH_SIZE,)           float32
) -> tf.Tensor:
    """
    One REINFORCE gradient step on the cloned M1 model.

    dean_cnn has a softmax output layer, so model output is a probability
    distribution over 7 columns.  We take log(clip(p, 1e-8, 1.0)) to get
    stable log-probabilities without applying a second softmax.
    """
    with tf.GradientTape() as tape:
        probs     = m1_model(boards, training=True)                        # (B, 7)
        log_probs = tf.math.log(tf.clip_by_value(probs, 1e-8, 1.0))       # (B, 7)

        # Gather the log-prob of the action that was actually taken
        batch_idx  = tf.range(tf.shape(cols)[0])                           # (B,)
        gather_idx = tf.stack([batch_idx, cols], axis=1)                   # (B, 2)
        chosen_log_probs = tf.gather_nd(log_probs, gather_idx)             # (B,)

        # REINFORCE loss: negate because we minimise (but want to maximise return)
        loss = -tf.reduce_mean(returns * chosen_log_probs)

    grads = tape.gradient(loss, m1_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, m1_model.trainable_variables))
    return loss

### Evaluation helper

In [11]:
def evaluate_win_rate(
    model,
    input_shape: tuple,
    eval_bots: list,
    games_per_opponent: int,
    eval_rng: random.Random,
) -> float:
    """
    Greedy evaluation: M1 plays argmax, each opponent plays its own argmax.
    Games alternate which side M1 plays to avoid first-mover bias.
    Returns overall win rate across all opponents combined.

    Using greedy play here (rather than stochastic) gives a cleaner signal:
    we want to measure the quality of the learned policy, not its randomness.
    """
    wins  = 0
    total = 0

    for opp in eval_bots:
        for game_idx in range(games_per_opponent):
            m1_player  = 1 if game_idx % 2 == 0 else -1
            m2_player  = -m1_player
            board      = np.zeros((6, 7), dtype=np.int8)
            cur_player = 1

            while True:
                legal = legal_moves(board)
                if not legal:
                    break   # draw — no win counted

                if cur_player == m1_player:
                    # M1: greedy argmax over legal moves
                    enc    = encode_board_for_model(board, m1_player, input_shape)
                    scores = model(enc, training=False).numpy().reshape(-1)
                    masked = np.full(7, -1e18, dtype=np.float64)
                    masked[legal] = scores[legal]
                    col = int(np.argmax(masked))
                else:
                    # Opponent: its own greedy argmax
                    col = opp.select_move(board, m2_player)

                board  = apply_move(board, col, cur_player)
                winner = check_winner(board)
                if winner is not None:
                    if winner == m1_player:
                        wins += 1
                    break

                cur_player *= -1

            total += 1

    return wins / total

## Training Loop

Each outer iteration:
1. Pick a random M2 from the opponent pool.
2. Play `GAMES_PER_ITER` games, collecting M1's `(board, col)` trajectory.
3. Build a fixed-size batch of 64 `(board, col, G_t)` triplets.
4. Take one REINFORCE gradient step.
5. Every `SNAPSHOT_EVERY` iters: add a frozen M1 copy to the pool.
6. Every `EVAL_EVERY` iters: record greedy win rate vs. eval opponents.

In [ ]:
# ── Tracking containers ───────────────────────────────────────────────────────
loss_history    = []   # list of (iteration, loss_value)
winrate_history = []   # list of (iteration, win_rate)

# Record baseline win rate before any training
print("Computing baseline win rate (untrained M1 clone) ...")
baseline_wr = evaluate_win_rate(
    m1_model, M1_INPUT_SHAPE, EVAL_OPPONENTS, EVAL_GAMES, rng
)
winrate_history.append((0, baseline_wr))
print(f"  Baseline win rate: {baseline_wr:.2%}")

Computing baseline win rate (untrained M1 clone) ...


In [ ]:
# ── Main policy gradient training loop ───────────────────────────────────────
print(f"Starting PG training for {NUM_ITERATIONS} iterations ...\n")

for iteration in range(1, NUM_ITERATIONS + 1):

    # ── Step 1: pick a random opponent from the pool ──────────────────────────
    m2_bot = rng.choice(opponent_pool)

    # ── Step 2: play GAMES_PER_ITER games and collect trajectories ────────────
    episode_batch = []
    for _ in range(GAMES_PER_ITER):
        traj, outcome = play_pg_game(
            m1_model,
            M1_INPUT_SHAPE,
            m2_bot,
            rng,
            opening_min=OPENING_MIN,
            opening_max=OPENING_MAX,
        )
        episode_batch.append((traj, outcome))

    # ── Step 3: build a fixed-size training batch ─────────────────────────────
    batch = build_training_batch(episode_batch, BATCH_SIZE, GAMMA)
    if batch is None:
        continue

    boards_b, cols_b, returns_b = batch

    # ── Step 4: one REINFORCE gradient step ───────────────────────────────────
    loss_val = pg_train_step(
        tf.constant(boards_b,  dtype=tf.float32),
        tf.constant(cols_b,    dtype=tf.int32),
        tf.constant(returns_b, dtype=tf.float32),
    )
    loss_history.append((iteration, float(loss_val)))

    # ── Step 5: snapshot M1 into the opponent pool ────────────────────────────
    if iteration % SNAPSHOT_EVERY == 0:
        snap_model = tf.keras.models.clone_model(m1_model)
        snap_model.set_weights(m1_model.get_weights())
        snap_name  = f"pg_m1_iter{iteration}"

        snap_bot = LoadedBot(
            name=snap_name,
            model=snap_model,
            model_path=MODELS_DIR / f"{snap_name}.keras",
            source="policy_gradient_snapshot",
            input_shape=M1_INPUT_SHAPE,
            role="pg_snapshot",
            notes=f"Frozen PG M1 snapshot at iteration {iteration}",
        )
        opponent_pool.append(snap_bot)

        # Trim oldest snapshot if pool exceeds cap (originals are never removed)
        while len(opponent_pool) > MAX_POOL_SIZE:
            for i, b in enumerate(opponent_pool):
                if b.role == "pg_snapshot":
                    opponent_pool.pop(i)
                    break

        print(f"  [iter {iteration:>4d}] Snapshot added: {snap_name}"
              f" | pool size: {len(opponent_pool)}/{MAX_POOL_SIZE}")

    # ── Step 6: periodic greedy evaluation ───────────────────────────────────
    if iteration % EVAL_EVERY == 0:
        wr = evaluate_win_rate(
            m1_model, M1_INPUT_SHAPE, EVAL_OPPONENTS, EVAL_GAMES, rng
        )
        winrate_history.append((iteration, wr))

        recent_losses = [l for _, l in loss_history[-EVAL_EVERY:]]
        avg_loss = np.mean(recent_losses)
        print(f"Iter {iteration:>4d}/{NUM_ITERATIONS}"
              f" | avg loss: {avg_loss:+.4f}"
              f" | win rate: {wr:.2%}"
              f" | pool size: {len(opponent_pool)}/{MAX_POOL_SIZE}")

print("\nTraining complete.")

Starting PG training for 600 iterations ...

Iter   25/600 | avg loss: -0.5121 | win rate: 62.50% | pool size: 4/15
Iter   50/600 | avg loss: -0.3791 | win rate: 37.50% | pool size: 4/15
  [iter   75] Snapshot added: pg_m1_iter75 | pool size: 5/15
Iter   75/600 | avg loss: -0.2586 | win rate: 37.50% | pool size: 5/15
Iter  100/600 | avg loss: -0.1472 | win rate: 37.50% | pool size: 5/15
Iter  125/600 | avg loss: -0.0857 | win rate: 50.00% | pool size: 5/15


KeyboardInterrupt: 

In [ ]:
wr_iters = [i for i, _ in winrate_history]
wr_vals  = [v for _, v in winrate_history]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(wr_iters, [v * 100 for v in wr_vals],
        marker="o", linewidth=2, color="steelblue",
        markersize=4, label="PG M1 win rate")
ax.axhline(wr_vals[0] * 100, linestyle="--", color="gray", linewidth=1,
           label=f"Baseline ({wr_vals[0]:.1%})")
ax.set_xlabel("Training Iteration")
ax.set_ylabel("Win Rate (%)")
ax.set_title("Policy Gradient — M1 Greedy Win Rate vs. Reference Opponents")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DATA_DIR / "pg_win_rate.png", dpi=150)
plt.show()
print(f"Saved to {DATA_DIR / 'pg_win_rate.png'}")

### Loss curve

In [ ]:
loss_iters = [i for i, _ in loss_history]
loss_vals  = [l for _, l in loss_history]

# Rolling average for readability (raw per-step loss is noisy)
window   = 20
smoothed = pd.Series(loss_vals).rolling(window, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(loss_iters, loss_vals, alpha=0.2, linewidth=0.8,
        color="coral", label="Raw loss per step")
ax.plot(loss_iters, smoothed, linewidth=2,
        color="darkred", label=f"Rolling mean (window={window})")
ax.set_xlabel("Training Iteration")
ax.set_ylabel("PG Loss")
ax.set_title("Policy Gradient — Training Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DATA_DIR / "pg_loss_curve.png", dpi=150)
plt.show()
print(f"Saved to {DATA_DIR / 'pg_loss_curve.png'}")

### Final head-to-head evaluation against all original opponents

50 greedy games per opponent (25 as first player, 25 as second).

In [ ]:
FINAL_EVAL_GAMES = 50
eval_rows = []

for opp in ORIGINAL_OPPONENTS:
    m1_wins = 0
    opp_wins = 0
    draws = 0

    for game_idx in range(FINAL_EVAL_GAMES):
        m1_player  = 1 if game_idx % 2 == 0 else -1
        m2_player  = -m1_player
        board      = np.zeros((6, 7), dtype=np.int8)
        cur_player = 1

        while True:
            legal = legal_moves(board)
            if not legal:
                draws += 1
                break

            if cur_player == m1_player:
                enc    = encode_board_for_model(board, m1_player, M1_INPUT_SHAPE)
                scores = m1_model(enc, training=False).numpy().reshape(-1)
                masked = np.full(7, -1e18, dtype=np.float64)
                masked[legal] = scores[legal]
                col = int(np.argmax(masked))
            else:
                col = opp.select_move(board, m2_player)

            board  = apply_move(board, col, cur_player)
            winner = check_winner(board)
            if winner is not None:
                if winner == m1_player:
                    m1_wins += 1
                else:
                    opp_wins += 1
                break

            cur_player *= -1

    eval_rows.append({
        "opponent":         opp.name,
        "pg_m1_wins":       m1_wins,
        "opponent_wins":    opp_wins,
        "draws":            draws,
        "pg_m1_win_rate":   m1_wins / FINAL_EVAL_GAMES,
    })

eval_df = pd.DataFrame(eval_rows).sort_values("pg_m1_win_rate", ascending=False)
eval_df

In [ ]:
# Bar chart: PG M1 win rate per opponent
fig, ax = plt.subplots(figsize=(9, 4))
colors = [
    "steelblue" if v >= 0.5 else "coral"
    for v in eval_df["pg_m1_win_rate"]
]
bars = ax.barh(
    eval_df["opponent"],
    eval_df["pg_m1_win_rate"] * 100,
    color=colors,
)
ax.axvline(50, linestyle="--", color="black", linewidth=1.2,
           label="50% break-even")
ax.set_xlabel("Win Rate (%)")
ax.set_title(
    f"PG-Trained M1 Win Rate vs. Each Original Opponent ({FINAL_EVAL_GAMES} games)"
)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend()
ax.grid(True, axis="x", alpha=0.3)
for bar, val in zip(bars, eval_df["pg_m1_win_rate"]):
    ax.text(
        bar.get_width() + 0.8,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.1%}",
        va="center", fontsize=9,
    )
plt.tight_layout()
plt.savefig(DATA_DIR / "pg_final_eval_bar.png", dpi=150)
plt.show()
print(f"Saved to {DATA_DIR / 'pg_final_eval_bar.png'}")

## Save Trained Model

In [ ]:
m1_model.save(PG_MODEL_PATH)
print(f"PG-trained M1 saved to: {PG_MODEL_PATH}")

# Summary statistics
final_wr    = winrate_history[-1][1]
baseline_wr = winrate_history[0][1]
improvement = final_wr - baseline_wr

print()
print(f"  Baseline win rate  : {baseline_wr:.2%}")
print(f"  Final win rate     : {final_wr:.2%}")
print(f"  Improvement        : {improvement:+.2%}")
print(f"  Training iterations: {NUM_ITERATIONS}")
print(f"  Total games played : ~{NUM_ITERATIONS * GAMES_PER_ITER:,}")
print(f"  Final pool size    : {len(opponent_pool)} bots")